# MNIST decision-only black-box benchmark: HSJA vs TtBA

**Exploratory reproduction notebook · Colab GPU recommended**

**Goal.** Compare **untargeted** HopSkipJumpAttack (HSJA, Foolbox) with **TtBA (ICML 2025)** on the **same correctly classified MNIST images**, with query cutoffs **100 / 500 / 1,000 / 3,000**. Save the **best queried adversarial image by L2** at each cutoff, report success under several L2 distortion limits, and export a **blind human-review template** for semantic validity.

**Important limitations.** (1) This is a small exploratory benchmark, **not evidence that SOTA attacks fail**. (2) Query counters include only attack-time model evaluations; clean-image eligibility screening occurs beforehand and is documented separately. (3) **Each image in a batch counts as one query**, not one API batch request. (4) The final model's numeric logits are used **only internally** to turn queries into predicted labels; neither decision-based method sees logits or gradients. (5) Human semantic labels are **not generated automatically**. (6) The TtBA author's code assumes CUDA and has hard-coded `.cuda()` operations; a GPU runtime is required for that method.

References: [HSJA](https://arxiv.org/abs/1904.02144) · [Foolbox](https://foolbox.readthedocs.io/en/stable/) · [TtBA paper](https://proceedings.mlr.press/v267/wang25fg.html) · [Official TtBA GitHub](https://github.com/BUPTAIOC/TtBA)

**To run:** Runtime → Change runtime type → **T4 GPU** → Run all. `N_IMAGES=5` is a quick smoke test; set it to `20` or `30` for your weekend report.

In [ ]:
# 1. Install the black-box libraries; keep Colab's torch/torchvision versions.
%pip -q install "foolbox==3.3.4" pandas matplotlib scipy

import sys, subprocess, os
from pathlib import Path
TTBA_DIR = Path('/content/TtBA')
if not TTBA_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/BUPTAIOC/TtBA.git',str(TTBA_DIR)],check=True)
print('TtBA source:', TTBA_DIR)
if TTBA_DIR.exists():
    rev = subprocess.run(['git','-C',str(TTBA_DIR),'rev-parse','HEAD'],capture_output=True,text=True)
    print('TtBA commit:', rev.stdout.strip())

In [ ]:
# 2. Settings — change only this cell for your main run.
import random, json, math, time, warnings, shutil
from types import SimpleNamespace
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
N_IMAGES = 5      # Smoke test: 5. For the report: 20 or 30.
QUERY_BUDGETS = (100, 500, 1000, 3000)
L2_LIMITS = (1.0, 2.0, 3.0, 5.0)
ATTACK_METHODS = ('HSJA', 'TtBA')
ONLY_DIGIT = None  # None: mix digits, or set to 1 for your earlier digit-1 experiments.
EPOCHS = 3
TRAIN_BATCH = 128
CHECKPOINT_PATH = None  # Optional: path to a saved SimpleCNN state_dict or {'model_state_dict': ...}
SAVE_DIR = Path('/content/blackbox_mnist_results')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', device, 'GPU required for official TtBA')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert tuple(sorted(QUERY_BUDGETS)) == tuple(QUERY_BUDGETS)

In [ ]:
# 3. Reproduce the same SimpleCNN ARCHITECTURE as your MNIST-CNN-PDPGD GitHub notebook.
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(32*7*7,128)
        self.fc2 = nn.Linear(128,10)
    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.reshape(x.shape[0], -1)
        return self.fc2(F.relu(self.fc1(x)))

train_ds = datasets.MNIST('/content/data',train=True,download=True,transform=transforms.ToTensor())
test_ds = datasets.MNIST('/content/data',train=False,download=True,transform=transforms.ToTensor())
train_loader=DataLoader(train_ds,batch_size=TRAIN_BATCH,shuffle=True,num_workers=2,pin_memory=(device.type=='cuda'))
test_loader=DataLoader(test_ds,batch_size=256,shuffle=False,num_workers=2,pin_memory=(device.type=='cuda'))

victim=SimpleCNN().to(device)
if CHECKPOINT_PATH is not None:
    state=torch.load(CHECKPOINT_PATH,map_location=device,weights_only=True)
    if isinstance(state,dict) and 'model_state_dict' in state: state=state['model_state_dict']
    victim.load_state_dict(state)
    print('Loaded checkpoint:', CHECKPOINT_PATH)
else:
    optim=torch.optim.Adam(victim.parameters(),lr=0.001)
    for epoch in range(EPOCHS):
        victim.train(); loss_sum=0.
        for xb,yb in train_loader:
            xb,yb=xb.to(device),yb.to(device)
            optim.zero_grad(set_to_none=True)
            loss=F.cross_entropy(victim(xb),yb)
            loss.backward(); optim.step()
            loss_sum+=float(loss.detach())
        print(f'Epoch {epoch+1}/{EPOCHS}: loss={loss_sum/len(train_loader):.4f}')
    torch.save(victim.state_dict(),SAVE_DIR/'simplecnn_mnist_state_dict.pt')

victim.eval()
for p in victim.parameters(): p.requires_grad_(False)
right=total=0
with torch.no_grad():
    for xb,yb in test_loader:
        preds=victim(xb.to(device)).argmax(1).cpu()
        right+=int((preds==yb).sum()); total+=len(yb)
print(f'ACTUAL MODEL TEST ACCURACY: {right}/{total} = {100*right/total:.2f}%')
print('Do not copy 98.58% from your earlier notebook unless you loaded THAT model/checkpoint.')

In [ ]:
# 4. Fix eligible examples BEFORE any attack; the same images go to both methods.
@torch.no_grad()
def select_eligible(ds, model, n, only_digit=None):
    selected=[]
    for idx,(x,y) in enumerate(ds):
        if only_digit is not None and int(y)!=int(only_digit): continue
        pred=int(model(x[None].to(device)).argmax(1).item())
        if pred==int(y):
            selected.append((idx,x[None].detach().cpu().clone(),int(y)))
        if len(selected)>=n: break
    if len(selected)<n: raise RuntimeError(f'Only {len(selected)} eligible examples found')
    return selected

cases=select_eligible(test_ds,victim,N_IMAGES,ONLY_DIGIT)
print('Selected test indices and true labels:',[(idx,label) for idx,_,label in cases])
# Screening queries are not included in per-attack Q; use the same predefined test set for both methods.

## Query oracle: count *every* attack-time image evaluation

All model evaluation goes through this wrapper. It exposes only `predict_label` to TtBA, and exposes logits to Foolbox's API solely so Foolbox can obtain `argmax` labels. The attack does **not** receive gradients or target scores. The wrapper records the best **actually queried and verified** adversarial image so far, making snapshots valid even when an attack is interrupted by the budget. **Budget exhaustion is raised before evaluating any additional image.**

In [ ]:
# 5. Instrumented model oracle, common to both methods.
class QueryBudgetExceeded(RuntimeError):
    pass

class QueryOracle(nn.Module):
    def __init__(self, base_model, original_cpu, original_label, budget, checkpoints):
        super().__init__()
        self.base_model=base_model
        self.x0=original_cpu.detach().cpu().clone()  # [1,1,28,28]
        self.y0=int(original_label)
        self.budget=int(budget)
        self.checkpoints=tuple(sorted(int(q) for q in checkpoints))
        self.num_queries=0
        self.best=None
        self.best_l2=float('inf')
        self.first_adv_query=None
        self.snapshots={}

    @torch.no_grad()
    def forward(self, images):
        if images.ndim==3: images=images.unsqueeze(0)
        n=int(images.shape[0]); remaining=self.budget-self.num_queries
        if remaining<=0: raise QueryBudgetExceeded('Target query budget exhausted')
        permitted=min(n,remaining)
        valid=images[:permitted].detach().clamp(0,1)
        # The base model is never used for generating gradients.
        logits=self.base_model(valid)
        labels=logits.argmax(1).detach().cpu().tolist()
        valid_cpu=valid.detach().cpu()
        for i,yi in enumerate(labels):
            self.num_queries+=1
            if int(yi)!=self.y0:
                if self.first_adv_query is None: self.first_adv_query=self.num_queries
                xi=valid_cpu[i:i+1].clone()
                d=float(torch.linalg.vector_norm((xi-self.x0).reshape(-1),ord=2).item())
                if d<self.best_l2:
                    self.best_l2=d
                    self.best=xi
            if self.num_queries in self.checkpoints:
                self.snapshots[self.num_queries]=self._capture()
        if permitted<n:
            # Some items in a batch were queried; the remainder were NOT.
            raise QueryBudgetExceeded('Budget exhausted during batched evaluation')
        return logits

    def predict_label(self, images):
        return self.forward(images).argmax(1)

    def _capture(self):
        return dict(queries_used=self.num_queries,
                    first_adv_query=self.first_adv_query,
                    adv=None if self.best is None else self.best.detach().cpu().clone(),
                    best_l2=None if self.best is None else float(self.best_l2))

    def finish(self):
        # If an attack terminates before reaching maximum Q, carry forward its last known state.
        current=self._capture()
        for q in self.checkpoints:
            if q not in self.snapshots:
                self.snapshots[q]=dict(current,adv=None if current['adv'] is None else current['adv'].clone())
        return self.snapshots

# Basic oracle test (does not call either attack).
class ToyClassifier(nn.Module):
    def forward(self,x):
        mean=x.flatten(1).mean(1)
        return torch.stack([0.5-mean,mean-0.5],dim=1)
toy=ToyClassifier()
o=QueryOracle(toy,torch.zeros(1,1,28,28),0,3,(1,2,3))
assert o.predict_label(torch.ones(1,1,28,28)).item()==1
try: o.predict_label(torch.ones(4,1,28,28))
except QueryBudgetExceeded: pass
assert o.num_queries==3 and set(o.snapshots)=={1,2,3}
print('Oracle accounting self-test PASSED')

In [ ]:
# 6. HSJA (Foolbox implementation, not a reimplementation).
import foolbox as fb

def run_hsja(oracle, x_cpu, y):
    fmodel=fb.PyTorchModel(oracle.to(device).eval(),bounds=(0.0,1.0),device=device)
    atk=fb.attacks.HopSkipJumpAttack(
        constraint='l2',
        steps=50,
        initial_gradient_eval_steps=24,
        max_gradient_eval_steps=256,
    )
    # The author library evaluates the classifier through oracle; only the oracle records candidates.
    try:
        _=atk.run(fmodel,x_cpu.to(device),torch.tensor([y],device=device,dtype=torch.int64))
    except QueryBudgetExceeded:
        pass
    return oracle.finish()

In [ ]:
# 7. TtBA (the author's implementation; requires CUDA, no substitute attack).
import importlib, sys
sys.path.insert(0,str(TTBA_DIR))
if device.type=='cuda':
    import TtBA
    print('Loaded official TtBA from:',TtBA.__file__)
else:
    print('TtBA not available on CPU: the official TtBA.py calls .cuda() directly.')

def run_ttba(oracle, x_cpu, y, image_idx):
    if device.type!='cuda':
        raise RuntimeError('Official TtBA requires a CUDA runtime. Enable GPU in Colab.')
    args=SimpleNamespace(
        budget=oracle.budget,
        epsilon=3.0, # Used only by author's internal success reporting, NOT our metric cutoff.
        early=0,
        binaryM=1,
        initDir=1,
    )
    attacker=TtBA.Attacker(
        args=args,
        model=oracle.to(device).eval(),
        im_orig=x_cpu.to(device),
        imgi=int(image_idx),
        attack_method='TtBA',
        tar_img=None,
        tar_label=None,
        iteration=1000,
        initial_query=30,
    )
    try:
        attacker.attack()
    except QueryBudgetExceeded:
        pass
    return oracle.finish()

## Run the benchmark

The single-run method is intentional: we run each method once to `max(QUERY_BUDGETS)` and snapshot the best **queried** adversarial image at intermediate budgets. This avoids rerunning a randomized attack from scratch for each budget. The two implementations may use different initializations and settings, so treat this as an **exploratory comparison**, not a formal head-to-head reproduction of each paper's official reported numbers.

In [ ]:
# 8. Run. Stores CSV rows AND images after each image, so a Colab interruption loses little work.
import contextlib, io
from PIL import Image

image_dir=SAVE_DIR/'adversarial_images';image_dir.mkdir(exist_ok=True)
results=[]

# Avoid notebooks accidentally reusing a stale file. Run this cell once per experiment setup.
if device.type!='cuda' and 'TtBA' in ATTACK_METHODS:
    raise RuntimeError('Switch Colab to a GPU runtime to compare both methods. No fake TtBA fallback.')

for case_i,(test_idx,x_cpu,true_y) in enumerate(cases):
    for meth in ATTACK_METHODS:
        random.seed(SEED+case_i); np.random.seed(SEED+case_i);torch.manual_seed(SEED+case_i)
        if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED+case_i)
        oracle=QueryOracle(victim,x_cpu,true_y,max(QUERY_BUDGETS),QUERY_BUDGETS).to(device)
        start=time.time()
        try:
            if meth=='HSJA': snaps=run_hsja(oracle,x_cpu,true_y)
            else:
                # The author code emits per-iteration progress; suppress it for a clean notebook.
                with contextlib.redirect_stdout(io.StringIO()):
                    snaps=run_ttba(oracle,x_cpu,true_y,test_idx)
        except Exception as e:
            print(f'ERROR {meth}, image {test_idx}: {type(e).__name__}: {e}')
            raise   # Don't silently turn implementation bugs into attack failures.
        used=oracle.num_queries
        for budget in QUERY_BUDGETS:
            s=snaps[budget]
            adv=s['adv']
            if adv is None:
                l2=linf=mse=float('nan'); success=False;filepath=''
            else:
                dx=(adv-x_cpu).float()
                l2=float(torch.linalg.vector_norm(dx.reshape(-1),ord=2))
                linf=float(dx.abs().max())
                mse=float(dx.square().mean())
                success=True  # oracle only stores actually queried images classified incorrectly
                filepath=str(image_dir/f'{meth}_idx{test_idx:05d}_Q{budget}.png')
                arr=(adv[0,0].numpy()*255).clip(0,255).astype(np.uint8)
                Image.fromarray(arr,mode='L').save(filepath)
            results.append(dict(method=meth,test_index=test_idx,true_label=true_y,
                                query_budget=budget,queries_used=min(used,budget),
                                has_adversarial_candidate=success,
                                first_adv_query=s['first_adv_query'],
                                l2=l2,linf=linf,mse=mse,adv_path=filepath))
        pd.DataFrame(results).to_csv(SAVE_DIR/'per_sample_results.csv',index=False)
        print(f'Image {case_i+1}/{len(cases)} | test idx={test_idx} label={true_y} | {meth} | actual target queries={used} | elapsed={time.time()-start:.1f}s | best L2={oracle.best_l2 if oracle.best is not None else None}')

raw=pd.DataFrame(results)
print('Saved:',SAVE_DIR/'per_sample_results.csv')
display(raw.head(10))

In [ ]:
# 9. Aggregate ASR(B, epsilon) for each L2 threshold, plus conditional distortion summaries.
raw=pd.read_csv(SAVE_DIR/'per_sample_results.csv')
rows=[]
for (method,q),g in raw.groupby(['method','query_budget'],sort=True):
    assert len(g)==len(cases),f'Missing results for {method}, Q={q}'
    for eps in L2_LIMITS:
        succeeded=g['has_adversarial_candidate'].astype(bool)&(g['l2']<=eps)
        rows.append({
            'method':method,'Q':q,'L2_limit':eps,
            'N_eligible':len(g),'successful_within_L2':int(succeeded.sum()),
            'ASR':float(succeeded.mean()),
            'median_L2_among_successes':float(g.loc[succeeded,'l2'].median()) if succeeded.any() else np.nan,
            'median_Linf_among_successes':float(g.loc[succeeded,'linf'].median()) if succeeded.any() else np.nan,
            'median_MSE_among_successes':float(g.loc[succeeded,'mse'].median()) if succeeded.any() else np.nan,
        })
summary=pd.DataFrame(rows)
summary.to_csv(SAVE_DIR/'summary_asr.csv',index=False)
display(summary)

plt.figure(figsize=(8,5))
for meth,gg in summary.groupby('method'):
    for eps in [2.0,3.0,5.0]:
        ss=gg[gg.L2_limit==eps].sort_values('Q')
        if len(ss):plt.plot(ss.Q,ss.ASR*100,marker='o',label=f'{meth}, L2<={eps:g}')
plt.xlabel('Target classifier queries per input')
plt.ylabel('Untargeted ASR (%)')
plt.title('MNIST: attack success vs query budget (exploratory)')
plt.xticks(QUERY_BUDGETS);plt.ylim(-3,103);plt.grid(alpha=0.25);plt.legend(fontsize=8)
plt.tight_layout();plt.savefig(SAVE_DIR/'asr_vs_queries.png',dpi=200);plt.show()

In [ ]:
# 10. Visual comparison. The model misclassification does NOT establish semantic preservation.
raw=pd.read_csv(SAVE_DIR/'per_sample_results.csv')
show_n=min(6,len(cases)); q=max(QUERY_BUDGETS)
fig,axes=plt.subplots(show_n,1+len(ATTACK_METHODS),figsize=(9,2.7*show_n),squeeze=False)
for ri,(idx,x0,y) in enumerate(cases[:show_n]):
    axes[ri,0].imshow(x0[0,0],cmap='gray',vmin=0,vmax=1)
    axes[ri,0].set_title(f'Original | y={y} | idx={idx}')
    for mi,meth in enumerate(ATTACK_METHODS,1):
        r=raw[(raw.test_index==idx)&(raw.method==meth)&(raw.query_budget==q)].iloc[0]
        if isinstance(r.adv_path,str) and Path(r.adv_path).exists():
            arr=np.array(Image.open(r.adv_path))
            axes[ri,mi].imshow(arr,cmap='gray',vmin=0,vmax=255)
            axes[ri,mi].set_title(f'{meth} | L2={r.l2:.2f}')
        else:
            axes[ri,mi].text(0.5,0.5,'No queried adversarial\nimage',ha='center',va='center',transform=axes[ri,mi].transAxes)
            axes[ri,mi].set_title(meth)
    for ax in axes[ri]: ax.axis('off')
plt.tight_layout();plt.savefig(SAVE_DIR/'original_vs_attacks.png',dpi=200);plt.show()

## Independent semantic review (manual; no invented scores)

A viewer should see a pair of **original + modified digit images** and record what digit the modified image represents, or `uncertain`. They should **not** see the attack name or the victim's prediction during review. The notebook exports blinded pairs and a template with empty labels. For a credible class-preservation study, seek at least two independent annotators and report disagreement/uncertainty.

**Do not interpret `L2` or classifier misclassification as a semantic verdict.** You may find that almost all attacked images retain their class; report that result if it occurs.

In [ ]:
# 11. Create blind visual-review cards, an annotator template, and a separate private manifest.
from PIL import ImageDraw, ImageFont
import uuid
review_dir=SAVE_DIR/'semantic_review_pairs';review_dir.mkdir(exist_ok=True)
manifest=[];review_template=[]
raw=pd.read_csv(SAVE_DIR/'per_sample_results.csv')
selected_lookup={idx:(x,y) for idx,x,y in cases}
for _,r in raw.iterrows():
    if not bool(r.has_adversarial_candidate): continue
    idx=int(r.test_index);orig,y=selected_lookup[idx]
    adv_path=Path(r.adv_path)
    if not adv_path.exists(): continue
    review_id=uuid.uuid4().hex[:12]
    original_img=Image.fromarray((orig[0,0].numpy()*255).clip(0,255).astype('uint8'),'L')
    adv_img=Image.open(adv_path).convert('L')
    scale=6; gap=30; canvas=Image.new('RGB',(2*28*scale+3*gap,28*scale+2*gap),'white')
    canvas.paste(original_img.resize((28*scale,28*scale),Image.Resampling.NEAREST),(gap,gap))
    canvas.paste(adv_img.resize((28*scale,28*scale),Image.Resampling.NEAREST),(2*gap+28*scale,gap))
    d=ImageDraw.Draw(canvas)
    d.text((gap,8),'Original',fill='black');d.text((2*gap+28*scale,8),'Modified',fill='black')
    out=review_dir/f'{review_id}.png';canvas.save(out)
    manifest.append({'review_id':review_id,'method':r.method,'test_index':idx,'true_label':y,
                     'query_budget':int(r.query_budget),'l2':float(r.l2),'pair_image_path':str(out)})
    review_template.append({'review_id':review_id,'digit_in_modified_image':'',
                            'uncertain_or_unrecognizable':'','notes':''})

pd.DataFrame(manifest).to_csv(SAVE_DIR/'review_manifest_PRIVATE.csv',index=False)
pd.DataFrame(review_template).to_csv(SAVE_DIR/'human_review_TEMPLATE.csv',index=False)
print('Review image pairs:',len(manifest))
print('Give reviewers the semantic_review_pairs/ folder and human_review_TEMPLATE.csv, not the private manifest.')

In [ ]:
# 12. OPTIONAL: after human labeling, compute semantic-valid ASR. This cell never invents human labels.
# Make a COPY of human_review_TEMPLATE.csv named human_review_FILLED.csv,
# fill digit_in_modified_image with 0..9; for ambiguous images put 'uncertain' in uncertain_or_unrecognizable.
filled_file=SAVE_DIR/'human_review_FILLED.csv'
if not filled_file.exists():
    print('No filled labels yet. SEMANTIC-VALID ASR IS NOT AVAILABLE. Human review template:',SAVE_DIR/'human_review_TEMPLATE.csv')
else:
    manifest=pd.read_csv(SAVE_DIR/'review_manifest_PRIVATE.csv')
    annotations=pd.read_csv(filled_file,dtype={'digit_in_modified_image':'string'})
    joined=manifest.merge(annotations,on='review_id',how='left',validate='one_to_one')
    joined['semantic_ok']=joined.apply(lambda r: int(r.digit_in_modified_image)==int(r.true_label)
         if pd.notna(r.digit_in_modified_image) and str(r.digit_in_modified_image).isdigit() else np.nan,axis=1)
    joined.to_csv(SAVE_DIR/'human_review_joined.csv',index=False)
    lines=[]
    # Correct denominator: ALL eligible inputs for each method and budget, not just successful cases.
    for (method,q),g in raw.groupby(['method','query_budget']):
        ann=joined[(joined.method==method)&(joined.query_budget==q)]
        for eps in L2_LIMITS:
            need=g[g.has_adversarial_candidate.astype(bool)&(g.l2<=eps)]
            ann_need=ann[ann.test_index.isin(need.test_index)]
            complete=len(ann_need)==len(need) and bool(ann_need.semantic_ok.notna().all())
            sem_asr=(float(ann_need.semantic_ok.astype(bool).sum())/len(g)) if complete else np.nan
            lines.append(dict(method=method,Q=q,L2_limit=eps,N_eligible=len(g),
                              N_attacks_needing_review=len(need),N_reviewed=int(ann_need.semantic_ok.notna().sum()),
                              semantic_valid_ASR=sem_asr))
    sem=pd.DataFrame(lines);display(sem)
    sem.to_csv(SAVE_DIR/'semantic_valid_asr.csv',index=False)
    print('NaN = incomplete annotation; no unverified semantic ASR will be reported.')

In [ ]:
# 13. Save reproducibility metadata and export all results as a zip file.
meta=dict(seed=SEED,N_images=len(cases),selected_test_indices=[i for i,_,_ in cases],
          budgets=list(QUERY_BUDGETS),l2_limits=list(L2_LIMITS),attack_methods=list(ATTACK_METHODS),
          threat_model='untargeted hard-label decision based; only argmax feedback exposed',
          dataset='MNIST',model='SimpleCNN from MNIST-CNN-PDPGD architecture',
          checkpoint=str(CHECKPOINT_PATH), torch_version=torch.__version__,
          foolbox_version=getattr(fb,'__version__','unknown'),
          screen_outside_budget=True,device=str(device))
with open(SAVE_DIR/'run_config.json','w') as f:json.dump(meta,f,indent=2)
archive=shutil.make_archive('/content/blackbox_mnist_results','zip',str(SAVE_DIR))
print('Results ZIP:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## How to report this to a professor

State **exactly** how many correctly classified images you tested, the actual CNN test accuracy, query limits, L2 limits, whether the attacks were targeted or untargeted, and which official implementations were used. Report `ASR(B, epsilon)` (with numerator/denominator), median L2 among successes, and independent semantic-valid ASR **only after the review is filled**. Explain that the test was an exploratory small-sample comparison with different initialization strategies, and does not prove any method is generally ineffective.

If official TtBA compatibility fails with newer Colab PyTorch, record the exception rather than silently replacing the method. The official source was designed for Python 3.11 / PyTorch 2.3 and contains device-specific assumptions. You can still run HSJA independently and note TtBA as incomplete. Never report fake comparison numbers.